<a href="https://colab.research.google.com/github/Saransh-Basu-01/50-Ml-Projects-to-understand-LLMs/blob/main/Project_33_QKV_weights_characteristics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np
import matplotlib.pyplot as plt

import torch
from transformers import AutoModelForCausalLM

# Part_1:A dictionary of models

## Task_1:
- Create two dictionaries, one called models and one called params.
Both dictionaries have four keys corresponding to four GPT-2 sizes:
small, medium, large, and xl. The corresponding value in the
models dictionary are the models themselves. This will allow you to
access, e.g., GPT-2-large using models[’large’].
The second dictionary (params) has the same keys as above,
but each item is itself another dictionary, which contains the
following keys: the number of transformer layers (n_layers), the
embeddings dimensionality (n_emb), the number of heads (n_heads),
and the head dimensionality (head_dim). You can find these
parameters in the .config property of each model.
Create and print a table of model sizes and their parameters
as shown below

In [2]:
# a list of lists of model infos
model_ids = [
    #  name    label
    ['small' ,       'gpt2'],
    ['medium','gpt2-medium'],
    ['large' , 'gpt2-large'],
    ['xl'    ,    'gpt2-xl']
]


# load all models into a dictionary
models = {}
params = {}

for modinfo in model_ids:

  # load the model
  models[modinfo[0]] = AutoModelForCausalLM.from_pretrained(modinfo[1])

  # count key parameters
  params[modinfo[0]] = {}
  params[modinfo[0]]['n_layers'] = models[modinfo[0]].config.n_layer
  params[modinfo[0]]['n_emb']    = models[modinfo[0]].config.n_embd
  params[modinfo[0]]['n_heads']  = models[modinfo[0]].config.n_head
  params[modinfo[0]]['head_dim'] = models[modinfo[0]].config.n_embd // params[modinfo[0]]['n_heads']


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

... (10 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 6.43GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [3]:
print('"models" keys:\n',models.keys())
print('\n"params[''small'']" keys:\n',params['small'].keys())

"models" keys:
 dict_keys(['small', 'medium', 'large', 'xl'])

"params[small]" keys:
 dict_keys(['n_layers', 'n_emb', 'n_heads', 'head_dim'])


In [4]:
print('  Model  | Embed.dim | Layers | n heads | head.dim')
print('---------+-----------+--------+---------+---------')
for name in models.keys():
  print(f" {name:>6}  |    {params[name]['n_emb']:4}   |   {params[name]['n_layers']}   |    {params[name]['n_heads']}   |   {params[name]['head_dim']}")


  Model  | Embed.dim | Layers | n heads | head.dim
---------+-----------+--------+---------+---------
  small  |     768   |   12   |    12   |   64
 medium  |    1024   |   24   |    16   |   64
  large  |    1280   |   36   |    20   |   64
     xl  |    1600   |   48   |    25   |   64


In [7]:
models['small'].transformer.h[5].attn.c_attn.weight

Parameter containing:
tensor([[ 0.1861, -0.1976,  0.0349,  ..., -0.2001, -0.0534,  0.0589],
        [-0.2022, -0.0133,  0.3201,  ..., -0.0700,  0.0005,  0.1399],
        [ 0.0021, -0.0444,  0.1136,  ..., -0.1196,  0.0206, -0.0324],
        ...,
        [-0.4044,  0.2771,  0.4169,  ...,  0.0580, -0.0181,  0.0009],
        [-0.1154,  0.0917,  0.1954,  ..., -0.0979,  0.1045, -0.0064],
        [ 0.1259,  0.1661,  0.2583,  ...,  0.1098, -0.0762, -0.0539]],
       requires_grad=True)

## Task_2:
- Count and report the total number of trainable attention parameters
in each model. The number of trainable parameters is the number of
weights plus the number of bias terms. Hint: All layers have exactly
the same sized weights matrices.

In [8]:

# count total number of parameters in attention subblock

for modlist in model_ids:

  name = modlist[0]

  # isolate one layer
  block = models[name].transformer.h[5].attn

  # count the parameters in this layer
  params_per_layer = (
    block.c_attn.weight.numel()+
    block.c_attn.bias.numel() +
    block.c_proj.weight.numel() +
    block.c_proj.bias.numel()
  )

  # total params is weights times layers
  totparams = params_per_layer * params[name]['n_layers']

  # and print the info
  print(f'{totparams/1e6:5,.1f}M attention weights in GPT2-{name}')

 28.3M attention weights in GPT2-small
100.8M attention weights in GPT2-medium
236.1M attention weights in GPT2-large
491.8M attention weights in GPT2-xl
